# 부록 03. Tool 데코레이터와 ToolRuntime

LangChain 1.0에서는 `@tool` 데코레이터와 `ToolRuntime`을 통해 강력한 도구를 정의할 수 있습니다.

## 학습 목표
- `@tool` 데코레이터 사용법 이해
- `ToolRuntime`으로 상태와 컨텍스트에 접근하는 방법
- 도구의 고급 기능 활용법 학습

## 1. 환경 설정

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정'}")

## 2. @tool 데코레이터 기본 사용법

`@tool` 데코레이터는 일반 Python 함수를 LangChain 도구로 변환합니다.

### 핵심 포인트
- 함수의 **docstring**이 도구의 설명이 됨 (모델이 언제 사용할지 결정하는 데 사용)
- **타입 힌트**가 필수 (입력 스키마 정의에 사용)

In [ ]:
from langchain.tools import tool

# 기본 도구 정의
@tool
def add_numbers(a: int, b: int) -> int:
    """두 숫자를 더합니다."""
    return a + b

# 도구 정보 확인
print(f"도구 이름: {add_numbers.name}")
print(f"도구 설명: {add_numbers.description}")
print(f"입력 스키마: {add_numbers.args_schema.model_json_schema()}")

In [ ]:
# 도구 직접 호출
result = add_numbers.invoke({"a": 5, "b": 3})
print(f"5 + 3 = {result}")

## 3. 도구 이름과 설명 커스터마이징

In [ ]:
# 이름 커스터마이징
@tool("calculator")
def my_calculator(expression: str) -> str:
    """수학 표현식을 계산합니다. 예: '2 + 3 * 4'"""
    try:
        # 주의: 실제 프로덕션에서는 eval() 사용을 피해야 함
        result = eval(expression)
        return f"결과: {result}"
    except Exception as e:
        return f"오류: {str(e)}"

print(f"도구 이름: {my_calculator.name}")
print(f"결과: {my_calculator.invoke({'expression': '2 + 3 * 4'})}")

In [ ]:
# 이름과 설명 모두 커스터마이징
@tool("web_search", description="인터넷에서 정보를 검색합니다. 최신 정보가 필요할 때 사용하세요.")
def search(query: str) -> str:
    """이 docstring은 사용되지 않습니다."""
    return f"'{query}'에 대한 검색 결과입니다. (시뮬레이션)"

print(f"도구 이름: {search.name}")
print(f"도구 설명: {search.description}")

## 4. Pydantic을 사용한 복잡한 입력 스키마

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, Literal

# Pydantic 모델로 입력 스키마 정의
class EmailInput(BaseModel):
    """이메일 전송을 위한 입력 스키마"""
    to: str = Field(description="수신자 이메일 주소")
    subject: str = Field(description="이메일 제목")
    body: str = Field(description="이메일 본문")
    priority: Literal["high", "normal", "low"] = Field(
        default="normal", 
        description="이메일 우선순위"
    )

@tool(args_schema=EmailInput)
def send_email(to: str, subject: str, body: str, priority: str = "normal") -> str:
    """이메일을 전송합니다."""
    return f"이메일 전송 완료: {to}에게 '{subject}' (우선순위: {priority})"

# 테스트
result = send_email.invoke({
    "to": "user@example.com",
    "subject": "테스트 이메일",
    "body": "안녕하세요!",
    "priority": "high"
})
print(result)

## 5. ToolRuntime으로 상태에 접근하기

`ToolRuntime`은 도구 내에서 에이전트의 상태, 컨텍스트, 스토어 등에 접근할 수 있게 해줍니다.

### ToolRuntime이 제공하는 것
- **state**: 현재 에이전트 상태 (메시지, 커스텀 필드 등)
- **context**: 런타임 컨텍스트 (사용자 정보 등 불변 데이터)
- **store**: 영구 저장소
- **stream_writer**: 실시간 스트리밍 출력
- **config**: 실행 설정
- **tool_call_id**: 현재 도구 호출 ID

In [ ]:
from langchain.tools import tool, ToolRuntime

# 상태에 접근하는 도구
@tool
def summarize_conversation(runtime: ToolRuntime) -> str:
    """지금까지의 대화를 요약합니다."""
    messages = runtime.state.get("messages", [])
    
    # 메시지 수 계산
    human_count = sum(1 for m in messages if hasattr(m, 'type') and m.type == 'human')
    ai_count = sum(1 for m in messages if hasattr(m, 'type') and m.type == 'ai')
    
    return f"대화 요약: 사용자 메시지 {human_count}개, AI 응답 {ai_count}개"

print(f"도구 생성됨: {summarize_conversation.name}")
print(f"설명: {summarize_conversation.description}")

## 6. 컨텍스트를 사용하는 도구

컨텍스트는 런타임에 주입되는 불변 데이터입니다. 사용자 정보, 세션 데이터 등을 전달할 때 유용합니다.

In [ ]:
from dataclasses import dataclass
from typing import TypedDict

# 컨텍스트 스키마 정의
@dataclass
class UserContext:
    """사용자 컨텍스트 정보"""
    user_id: str
    user_name: str
    is_premium: bool = False

# 컨텍스트를 사용하는 도구
@tool
def get_user_info(runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자의 정보를 조회합니다."""
    ctx = runtime.context
    premium_status = "프리미엄 회원" if ctx.is_premium else "일반 회원"
    return f"사용자: {ctx.user_name} (ID: {ctx.user_id}) - {premium_status}"

print("컨텍스트 기반 도구가 정의되었습니다.")

## 7. Command를 사용한 상태 업데이트

도구에서 `Command`를 반환하면 에이전트 상태를 직접 업데이트할 수 있습니다.

In [ ]:
from langgraph.types import Command

@tool
def set_user_preference(preference: str, value: str, runtime: ToolRuntime) -> Command:
    """사용자 선호도를 설정합니다."""
    # 현재 선호도 가져오기
    current_prefs = runtime.state.get("user_preferences", {})
    
    # 새 선호도 추가
    updated_prefs = {**current_prefs, preference: value}
    
    # Command로 상태 업데이트 반환
    return Command(
        update={"user_preferences": updated_prefs}
    )

print("상태 업데이트 도구가 정의되었습니다.")

## 8. 실제 에이전트에서 도구 사용하기

In [ ]:
from langchain.agents import create_agent
from datetime import datetime

# 실용적인 도구들 정의
@tool
def get_current_time() -> str:
    """현재 날짜와 시간을 반환합니다."""
    return datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분 %S초")

@tool
def calculate(expression: str) -> str:
    """수학 계산을 수행합니다. 예: '15 * 4 + 10'"""
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"

@tool
def get_weather(city: str) -> str:
    """도시의 날씨 정보를 조회합니다."""
    # 시뮬레이션 데이터
    weather_data = {
        "서울": "맑음, 기온 22°C",
        "부산": "흐림, 기온 20°C",
        "제주": "비, 기온 18°C"
    }
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

# 에이전트 생성
agent = create_agent(
    model="gpt-4o-mini",
    tools=[get_current_time, calculate, get_weather],
    system_prompt="당신은 도움이 되는 어시스턴트입니다. 필요한 경우 도구를 사용하세요."
)

print("에이전트가 생성되었습니다.")

In [ ]:
# 에이전트 실행 테스트
test_questions = [
    "지금 몇 시야?",
    "서울 날씨 어때?",
    "125 곱하기 8은 얼마야?"
]

for question in test_questions:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(f"Q: {question}")
    print(f"A: {result['messages'][-1].content}")
    print("-" * 50)

## 9. 비동기 도구

비동기 함수도 도구로 정의할 수 있습니다.

In [ ]:
import asyncio

@tool
async def async_search(query: str) -> str:
    """비동기로 검색을 수행합니다."""
    # 네트워크 요청 시뮬레이션
    await asyncio.sleep(0.5)
    return f"'{query}'에 대한 비동기 검색 결과입니다."

# 비동기 도구 테스트
result = await async_search.ainvoke({"query": "LangChain 튜토리얼"})
print(result)

## 10. 요약

### 이 노트북에서 배운 것

1. **@tool 데코레이터 기본**
   - docstring이 도구 설명이 됨
   - 타입 힌트로 입력 스키마 정의

2. **도구 커스터마이징**
   - 이름과 설명 지정: `@tool("name", description="...")`
   - Pydantic 모델로 복잡한 스키마 정의

3. **ToolRuntime**
   - `runtime.state`: 에이전트 상태 접근
   - `runtime.context`: 런타임 컨텍스트 접근
   - `runtime.store`: 영구 저장소 접근

4. **Command로 상태 업데이트**
   - `Command(update={...})`로 상태 변경

5. **비동기 도구**
   - `async def`로 비동기 도구 정의 가능

### 다음 단계
- 부록_04: create_agent로 간단한 에이전트 만들기